In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("../data/processed")

short_path = DATA / "cases_v3_short.csv"
full_path = DATA / "cases_v3_full.csv"
coded_path = DATA / "cases_court_level_coded.csv"

df_short = pd.read_csv(short_path)
df_full = pd.read_csv(full_path)
df_coded = pd.read_csv(coded_path)

print("short:", df_short.shape)
print("full:", df_full.shape)
print("coded:", df_coded.shape)

new_cols = ["court_level"]
key = ["case_id"]

df_coded_subset = df_coded[key + new_cols].copy()

print("Duplicate case_id in coded:", df_coded_subset["case_id"].duplicated().sum())
print("Duplicate case_id in short:", df_short["case_id"].duplicated().sum())
print("Duplicate case_id in full:", df_full["case_id"].duplicated().sum())

coded_not_in_short = sorted(set(df_coded_subset["case_id"]) - set(df_short["case_id"]))
short_not_in_coded = sorted(set(df_short["case_id"]) - set(df_coded_subset["case_id"]))

print("coded not in short:", len(coded_not_in_short))
print(coded_not_in_short[:10])
print("short not in coded:", len(short_not_in_coded))
print(short_not_in_coded[:10])

short: (1075, 104)
full: (1075, 636)
coded: (1077, 14)
Duplicate case_id in coded: 0
Duplicate case_id in short: 0
Duplicate case_id in full: 0
coded not in short: 2
['European Union_2009_01', 'European Union_2020_01']
short not in coded: 0
[]


In [12]:
cases_v3_short_plus = df_short.merge(
    df_coded_subset,
    on="case_id",
    how="left",
    validate="one_to_one"
)

cases_v3_full_plus = df_full.merge(
    df_coded_subset,
    on="case_id",
    how="left",
    validate="one_to_one"
)

print(cases_v3_short_plus.shape)
print(cases_v3_full_plus.shape)

cases_v3_short_plus[new_cols].isna().sum()

(1075, 105)
(1075, 637)


court_level    3
dtype: int64

In [13]:
cases_v3_short_plus["court_level"].value_counts(dropna=False)

court_level
low               325
high              296
mid               205
reg               186
oversight body     56
intl                4
NaN                 3
Name: count, dtype: int64

In [14]:
for df in [cases_v3_short_plus, cases_v3_full_plus]:
    df["high_court"] = (df["court_level"] == "high").astype(int)
    
    df["low_court"] = df["court_level"].isin(["low", "mid"]).astype(int)
    
    df["supranatl_court"] = df["court_level"].isin(
        ["reg", "oversight body", "intl"]
    ).astype(int)

In [15]:
for col in ["high_court", "low_court", "supranatl_court"]:
    print("\n", col)
    print(cases_v3_short_plus[col].value_counts())


 high_court
high_court
0    779
1    296
Name: count, dtype: int64

 low_court
low_court
0    545
1    530
Name: count, dtype: int64

 supranatl_court
supranatl_court
0    829
1    246
Name: count, dtype: int64


In [16]:
cases_v3_short_plus.to_csv(DATA / "cases_v3_short_courtlevel.csv", index=False)
cases_v3_full_plus.to_csv(DATA / "cases_v3_full_courtlevel.csv", index=False)

In [17]:
for df in [cases_v3_short_plus, cases_v3_full_plus]:
    
    df["j_ind"] = np.where(
        df["high_court"] == 1,
        df["v2juhcind"],
        np.where(
            df["low_court"] == 1,
            df["v2juncind"],
            np.nan
        )
    )

    df["j_ind_lag1"] = np.where(
        df["high_court"] == 1,
        df["v2juhcind_lag1"],
        np.where(
            df["low_court"] == 1,
            df["v2juncind_lag1"],
            np.nan
        )
    )

    df["j_ind_lag2"] = np.where(
        df["high_court"] == 1,
        df["v2juhcind_lag2"],
        np.where(
            df["low_court"] == 1,
            df["v2juncind_lag2"],
            np.nan
        )
    )

In [18]:
cases_v3_short_plus[["j_ind", "j_ind_lag1", "j_ind_lag2"]].describe()

,j_ind,j_ind_lag1,j_ind_lag2
count,826.000000,826.000000,826.000000
mean,1.565423,1.588758,1.598222
std,1.182439,1.180482,1.156197
min,-2.685000,-2.685000,-2.765000
25%,0.994000,1.007000,1.035250
50%,2.184000,2.184000,2.002000
75%,2.366000,2.366000,2.366000
max,3.366000,3.366000,3.366000


In [19]:
cases_v3_short_plus.groupby("court_level")[[
    "j_ind", "j_ind_lag1", "j_ind_lag2"
]].count()

,j_ind,j_ind_lag1,j_ind_lag2
court_level,,,
high,296,296,296
intl,0,0,0
low,325,325,325
mid,205,205,205
oversight body,0,0,0
reg,0,0,0


In [20]:
cases_v3_short_plus.to_csv(DATA / "cases_v3_short_merged.csv", index=False)
cases_v3_full_plus.to_csv(DATA / "cases_v3_full_merged.csv", index=False)

In [21]:
cases_v3_short_plus.groupby("court_level")["j_ind"].mean()

court_level
high              1.592591
intl                   NaN
low               1.344474
mid               1.876478
oversight body         NaN
reg                    NaN
Name: j_ind, dtype: float64